In [ ]:
# --- Rutes de l'ONOFRE: les defineix publish.sh a partir d'onofre.env ---
# Si no estan definides, es fan servir les rutes d'abans de la migració.
import os
from pathlib import Path
ONOFRE_REPO  = os.environ.get("ONOFRE_REPO")  or str(Path.cwd().parent)
ONOFRE_REF   = os.environ.get("ONOFRE_REF")   or "/home/pguarque/dades_referencia"
ONOFRE_ESTAT = os.environ.get("ONOFRE_ESTAT") or "/home/pguarque/prova_ICON_simu"
ONOFRE_MAPES = os.environ.get("ONOFRE_MAPES") or "/home/pguarque/outputs"


In [ ]:
import geopandas as gpd
import pandas as pd
import xarray as xr
import numpy as np

import datetime as dt
from datetime import datetime, timedelta, date

import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import ScalarMappable
import matplotlib.gridspec as gridspec
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds
from scipy.spatial import cKDTree
from shapely.geometry import Polygon

import requests
import tempfile
import os
from pathlib import Path
import bz2
import cfgrib
import glob, re

import metpy.calc as mpcalc
from metpy.units import units

import warnings
warnings.filterwarnings("ignore")

OUTPUT_PLOTS = ONOFRE_MAPES

regions = gpd.read_file(f"{ONOFRE_REF}/RE_bombers/re_bombers_proj.shp")
regions = regions.to_crs("EPSG:4326")
regions_feat = cfeature.ShapelyFeature(regions['geometry'], ccrs.PlateCarree(), facecolor='none')

comarques = gpd.read_file(f"{ONOFRE_REF}/comarques/comarques_cat.shp")
comarques = comarques.to_crs("EPSG:4326")
comarques_feat = cfeature.ShapelyFeature(comarques['geometry'], ccrs.PlateCarree(), facecolor='none')

inv_catalunya = gpd.read_file(f"{ONOFRE_REF}/comarques/no_cat.shp")
inv_catalunya = inv_catalunya.to_crs("EPSG:4326")
invCAT_feat = cfeature.ShapelyFeature(inv_catalunya['geometry'], ccrs.PlateCarree(), facecolor='none')

SMC_ROOT     = "/home/labfire/data/SMC-STATIONS"
SMC = dict(temp=32, hr=33, precip=35, rad=36, tmax=40, tmin=42, hrmin=44)

In [ ]:
# ── Configuració ──────────────────────────────────────────────────────────────
BASE_URL_DWD   = "https://opendata.dwd.de/weather/nwp/icon-eu/grib"
BASE_URL_EFFIS = "https://maps.effis.emergency.copernicus.eu/gwis"

RUN_DATE       = dt.date.today()
RUN_HOUR       = int(os.environ.get("ONOFRE_RUN_HOUR", 0))  # 0 o 12 — configurable des de fora (ex: publish.sh)

LLINDAR_SWE_MM = 10.0

LONS_ICON = np.arange(-23.5, 62.376, 0.0625)
LATS_ICON = np.arange( 29.5, 70.376, 0.0625)


# ── 1. Descàrrega DC (EFFIS WMS) ──────────────────────────────────────────────
def descarrega_dc_effis(data, max_retries=3, backoff=5):
    """
    Descarrega el DC des de GWIS/EFFIS amb reintents i validació real del
    contingut. Si falla després de tots els intents, retorna None en comptes
    de petar — el DC d'aquell dia queda com a NaN i la resta del pipeline
    continua (en comptes d'aturar tot el notebook per un servidor extern caigut).
    """
    import time

    params = {
        "LAYERS":      "ecmwf.dc",
        "FORMAT":      "image/tiff",
        "TRANSPARENT": "true",
        "SINGLETILE":  "false",
        "SERVICE":     "WMS",
        "VERSION":     "1.1.1",
        "REQUEST":     "GetMap",
        "STYLES":      "",
        "SRS":         "EPSG:4326",
        "BBOX":        "-23.5,29.5,62.375,70.375",
        "WIDTH":       "1374",
        "HEIGHT":      "655",
        "TIME":        data.isoformat(),
    }
    print(f"Descarregant DC ECMWF/GWIS per a {data.isoformat()}...")

    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(BASE_URL_EFFIS, params=params, timeout=30)
            r.raise_for_status()
            content = r.content

            if len(content) < 1000:
                raise ValueError(f"Resposta massa petita ({len(content)} bytes) — probable error del servidor")
            if b"xml" in content[:200].lower() or b"<html" in content[:200].lower():
                raise ValueError(f"El servidor ha retornat un error en comptes d'un TIFF: {content[:200]!r}")
            if content[:4] not in (b"II*\x00", b"MM\x00*"):
                raise ValueError("El contingut no té els magic bytes d'un TIFF vàlid")

            nom_tiff = tempfile.mktemp(suffix=".tiff", dir="/tmp")
            with open(nom_tiff, "wb") as f:
                f.write(content)

            # verificació final: que rasterio el pugui obrir de veritat
            with rasterio.open(nom_tiff):
                pass

            print(f"  ✓ TIFF temporal: {nom_tiff}")
            return nom_tiff

        except Exception as e:
            print(f"  ⚠ Intent {attempt}/{max_retries} fallit: {e}")
            if attempt < max_retries:
                time.sleep(backoff)

    print(f"  ✗ No s'ha pogut descarregar el DC després de {max_retries} intents. "
          f"S'omet aquest factor per avui (DC quedarà com a NaN).")
    return None


# ── 2. Descàrrega i càrrega SWE (tot en memòria) ─────────────────────────────
def descarrega_i_carrega_swe(data, hora_run=RUN_HOUR):
    run_str = dt.datetime.combine(data, dt.time(hora_run)).strftime("%Y%m%d%H")
    run_hh  = f"{hora_run:02d}"
    nom_fitxer = (
        f"icon-eu_europe_regular-lat-lon_single-level_"
        f"{run_str}_000_W_SNOW.grib2.bz2"
    )
    url = f"{BASE_URL_DWD}/{run_hh}/w_snow/{nom_fitxer}"

    print(f"Descarregant SWE (W_SNOW) a memòria...")
    r = requests.get(url, timeout=60)
    r.raise_for_status()

    grib2_bytes = bz2.decompress(r.content)

    with tempfile.NamedTemporaryFile(suffix=".grib2", delete=True, dir="/tmp") as tmp:
        tmp.write(grib2_bytes)
        tmp.flush()
        ds = xr.open_dataset(tmp.name, engine="cfgrib").load()

    var = list(ds.data_vars)[0]
    swe = ds[var]
    print(f"  Variable SWE: '{var}'  min={float(swe.min()):.3f}  max={float(swe.max()):.3f} kg/m²")
    return swe


# ── 3. Reprojecta DC a malla ICON-EU ─────────────────────────────────────────
def reprojecta_dc(tiff):
    n_lats = len(LATS_ICON)
    n_lons = len(LONS_ICON)

    if tiff is None:
        print("  ⚠ Sense TIFF de DC vàlid — graella de DC buida (NaN) per avui.")
        return np.full((n_lats, n_lons), np.nan, dtype=np.float32)

    transform_desti = from_bounds(
        LONS_ICON[0], LATS_ICON[0],
        LONS_ICON[-1], LATS_ICON[-1],
        n_lons, n_lats
    )

    try:
        with rasterio.open(tiff) as src:
            dc_raw = src.read(1).astype(np.float32)
            transform_origen = src.transform
            crs_origen       = src.crs
    except Exception as e:
        print(f"  ✗ Error obrint el TIFF de DC ({e}). Graella de DC buida (NaN) per avui.")
        try:
            os.unlink(tiff)
        except Exception:
            pass
        return np.full((n_lats, n_lons), np.nan, dtype=np.float32)

    dc_raw[dc_raw > 5000] = np.nan
    dc_raw[dc_raw < 0]    = np.nan

    dc_icon = np.full((n_lats, n_lons), np.nan, dtype=np.float32)
    reproject(
        source=dc_raw,
        destination=dc_icon,
        src_transform=transform_origen,
        src_crs=crs_origen,
        dst_transform=transform_desti,
        dst_crs="EPSG:4326",
        resampling=Resampling.bilinear,
        src_nodata=np.nan,
        dst_nodata=np.nan,
    )
    dc_icon = dc_icon[::-1, :]
    print(f"  DC reprojectat — min: {np.nanmin(dc_icon):.1f}  max: {np.nanmax(dc_icon):.1f}")

    os.unlink(tiff)  # esborra el TIFF temporal
    return dc_icon


# ── 4. Aplica màscara de neu ──────────────────────────────────────────────────
def aplica_mascara_neu(dc_icon, swe, llindar=LLINDAR_SWE_MM):
    swe_da = xr.DataArray(
        swe.values,
        dims=["lat", "lon"],
        coords={
            "lat": swe.latitude.values  if "latitude"  in swe.coords else swe.lat.values,
            "lon": swe.longitude.values if "longitude" in swe.coords else swe.lon.values,
        }
    ).sortby("lat").sortby("lon")

    swe_icon = swe_da.interp(lat=LATS_ICON, lon=LONS_ICON, method="linear")

    mascara = swe_icon.values > llindar
    n_px    = int(mascara.sum())
    print(f"  Píxels emmascarats per neu (SWE>{llindar}mm): {n_px} ({100*n_px/mascara.size:.1f}%)")

    dc_mascat = dc_icon.copy()
    dc_mascat[mascara] = np.nan
    return dc_mascat, swe_icon.values


# ── Execució principal ────────────────────────────────────────────────────────
if __name__ == "__main__":
    data = RUN_DATE

    print("=== 1. DC (EFFIS) ===")
    tiff = descarrega_dc_effis(data)

    print("\n=== 2. SWE (ICON-EU) ===")
    swe = descarrega_i_carrega_swe(data)

    print("\n=== 3. Reprojecció DC → ICON-EU ===")
    dc_icon = reprojecta_dc(tiff)

    print("\n=== 4. Màscara de neu ===")
    dc_mascat, swe_icon = aplica_mascara_neu(dc_icon, swe)

    # dc_mascat → array (657, 1377) float32, NaN on neu
    # swe_icon  → array (657, 1377) float32, kg/m²
    print("\n✓ Llest. dc_mascat i swe_icon disponibles per als càlculs.")

In [ ]:
rt  = datetime(RUN_DATE.year, RUN_DATE.month, RUN_DATE.day, RUN_HOUR)
tag = rt.strftime("%Y%m%d_%H")

surf = f'/home/labfire/data/ICON-EU/ICONEU_surface_{tag}.nc'
ds_surf = xr.open_dataset(surf)
print('nom: ds_surf')
print(ds_surf)

print('==================================')

mod = f'/home/labfire/data/ICON-EU/ICONEU_model_{tag}.nc'
ds_per = xr.open_dataset(mod)
print('nom: ds_per')
print(ds_per)

print('==================================')

lat_cat = ds_surf.latitude.values
lon_cat = ds_surf.longitude.values
doy = rt.timetuple().tm_yday

print('==================================')

# DC des de memòria (dc_mascat calculat a la cel·la anterior)
ds_DC = xr.DataArray(
    dc_mascat,
    dims=["lat", "lon"],
    coords={"lat": LATS_ICON, "lon": LONS_ICON},
    name="DC"
).to_dataset()
ds_DC_cat = ds_DC.sel(lat=lat_cat, lon=lon_cat, method="nearest")
print('nom: ds_DC_cat')
print(ds_DC_cat)
print('==================================')

percentils = "/home/labfire/data/GRAF_MAPS/static_data/DC/climatologia_dc_icon_eu_bo.nc"
ds_percentils = xr.open_dataset(percentils)
ds_perc_cat = ds_percentils.sel(
    lat=lat_cat,
    lon=lon_cat,
    dayofyear=doy,
    method="nearest"
)
print('nom: ds_perc_cat')
print(ds_perc_cat)


# ── Domini de càlcul ────────────────────────────────────────────────────────
# EUROPE_MODE=True  -> tot el domini ICON-EU carregat, sense retallar
# EUROPE_MODE=False -> comportament original, només Catalunya
EUROPE_MODE = True

if EUROPE_MODE:
    # Alinea el DC (graella pròpia, no exactament igual a la d'ICON-EU) a
    # la graella real d'ds_surf/ds_per amb "nearest" — mateix que fèiem
    # pel cas Catalunya, ara sobre lat_cat/lon_cat sencers (ja no retallats).
    ds_DC_cat = ds_DC.sel(lat=lat_cat, lon=lon_cat, method="nearest")
    ds_perc_cat = ds_percentils.sel(
        lat=lat_cat,
        lon=lon_cat,
        dayofyear=doy,
        method="nearest"
    )
else:
    lat_min, lat_max = 40.375, 43.0
    lon_min, lon_max = 0.0, 3.5
    # França
    #lat_min, lat_max = 42, 46
    #lon_min, lon_max = 1.25, 8.25
    # Aragó
    #lat_min, lat_max = 39.85, 43.5
    #lon_min, lon_max = -2.25, 1
    #lat_min, lat_max = 37.5, 39.5
    #lon_min, lon_max = 23, 24.5

    ds_surf = ds_surf.sel(latitude=slice(lat_min, lat_max), longitude=slice(lon_min, lon_max))
    ds_per  = ds_per.sel( latitude=slice(lat_min, lat_max), longitude=slice(lon_min, lon_max))
    ds_DC_cat = ds_DC.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
    ds_perc_cat = ds_percentils.sel(
        lat=slice(lat_min, lat_max),
        lon=slice(lon_min, lon_max)
    ).sel(
        dayofyear=doy,
        method="nearest"
    )

# Alinear temps: agafar del surface només els passos que coincideixen amb model
common_times = np.intersect1d(ds_surf.time.values, ds_per.time.values)
ds_surf = ds_surf.sel(time=common_times)

In [ ]:
# CANVI DIRECCIÓ DE VENT
def reclassify_wind_change(value):
    abs_value = abs(value)
    if abs_value <= 30:
        category = 0
    elif 30 < abs_value <= 60:
        category = 1
    elif 60 < abs_value <= 90:
        category = 2
    elif 90 < abs_value <= 120:
        category = 3
    elif 120 < abs_value <= 150:
        category = 4
    elif 150 < abs_value <= 180:
        category = 5
    elif 180 < abs_value <= 210:
        category = 5
    elif 210 < abs_value <= 240:
        category = 4
    elif 240 < abs_value <= 270:
        category = 3
    elif 270 < abs_value <= 300:
        category = 2
    elif 300 < abs_value <= 330:
        category = 1
    elif 330 < abs_value <= 360:
        category = 0
    else:
        # Handle cases outside of the expected range
        category = -1
    return category


# CLASSIFICACIÓ INESTABILITAT
# def calcul_gradients(gradient_surface_to_abl, gradient_abl_to_2000m):
#     gradient_surface_to_abl = np.asarray(gradient_surface_to_abl)
#     gradient_abl_to_2000m = np.asarray(gradient_abl_to_2000m)
    
#     # Define the conditions for the pyroconvection types
#     if gradient_surface_to_abl <= 0.003 and gradient_abl_to_2000m <= 0.005:
#         perfil = 1
#     if gradient_surface_to_abl <= 0.003 and gradient_abl_to_2000m > 0.005:
#         perfil = 2
#     if gradient_surface_to_abl > 0.003 and gradient_abl_to_2000m <= 0.005:
#         perfil = 3
#     if gradient_surface_to_abl > 0.003 and gradient_abl_to_2000m > 0.005:
#         perfil = 4

#     return perfil

def calcul_gradients(gradient_surface_to_abl, gradient_abl_to_2000m):
    gradient_surface_to_abl = np.asarray(gradient_surface_to_abl)
    gradient_abl_to_2000m = np.asarray(gradient_abl_to_2000m)
    perfil = 1  # valor per defecte (tria el que toqui al teu esquema)

    if gradient_surface_to_abl > 0.003 and gradient_abl_to_2000m > 0.005:
        perfil = 4
    elif gradient_surface_to_abl > 0.003 and gradient_abl_to_2000m <= 0.005:
        perfil = 3
    elif gradient_surface_to_abl <= 0.003 and gradient_abl_to_2000m > 0.005:
        perfil = 2
    # else: perfil queda com el default

    return perfil



# CLASSIFICACIÓ COMPLEXITAT
def calculate_Finestra_IF(wind_speed, durations, da_DewP_celsius, wind_run, Tipus_pyroc, da_Ac_Pluja):
    Finestra_IF = np.zeros_like(wind_speed, dtype=float)  # Initialize Finestra_IF with zeros of the same shape, but as floats

    # Check wind_speed conditions
    Finestra_IF += (20 < wind_speed) & (wind_speed <= 40)  # Add 1 where condition is met
    Finestra_IF += 2 * (wind_speed > 40)  # Add 2 where condition is met

    # Check da_HR conditions
    Finestra_IF += (6 <= durations) & (durations < 12)  # Add 1 where condition is met
    Finestra_IF += 2 * ((durations >= 12) & (durations < 24)) # Add 2 where condition is met
    Finestra_IF += 3 * (durations > 24)

    # Check da_DewP conditions
    Finestra_IF += (da_DewP_celsius < 7)  # Add 0.5 where condition is met

    # Check wind_run condition
    Finestra_IF += ((wind_run > 1) & (wind_run < 2.5))  # Add 0.5 where condition is met
    Finestra_IF += (wind_run >= 2.5)  # Add 1 where condition is met
    
    # Check Tipus_pyroc condition
    Finestra_IF += ((Tipus_pyroc == 1) | (Tipus_pyroc == 2))  # Add 0.5 where condition is met

    # Check da_AcPluja condition
    #Finestra_IF = np.where(da_Ac_Pluja > 12, 0, Finestra_IF)  # Set to 0 where condition is met

    return Finestra_IF


# RECLASSIFICACIÓ COMPLEXITAT
def finestra_reclass(Finestra_IF):
    Finestra_IF = np.asarray(Finestra_IF)

    complexitat = np.zeros_like(Finestra_IF)

    complexitat[Finestra_IF == 0] = 0
    complexitat[Finestra_IF == 1] = 1
    complexitat[Finestra_IF == 2] = 2
    complexitat[(Finestra_IF >= 3) & (Finestra_IF <= 4)] = 3
    complexitat[(Finestra_IF >= 5) & (Finestra_IF <= 6)] = 4
    complexitat[Finestra_IF == 7] = 5
    complexitat[Finestra_IF == 8] = 6

    return complexitat


# TIPUS PIROCONVECCIÓ
def tipus_piroconveccio(ABLgradient, FTgradient, da_ABLh, LCL_ABL):
    ABLgradient = np.asarray(ABLgradient)
    FTgradient = np.asarray(FTgradient)
    
    # Situació IN-IN
    if ABLgradient <= 0.0035 and FTgradient <= 0.005:
        if LCL_ABL < 1:
            return 1
        elif 1 <= LCL_ABL < 1.5:
            return 2
        elif 1.5 <= LCL_ABL < 3:
            return 3
        elif LCL_ABL >= 3:
            return 4

    # Situació IN-ES
    if ABLgradient <= 0.0035 and FTgradient > 0.005:
        if da_ABLh < 2000 and LCL_ABL < 1.5:
            return 3
        else:
            return 4
    
    # Situació ES-IN (alta intestabilitat)
    if ABLgradient > 0.0035 and FTgradient < 0.003:
        if da_ABLh < 1500 and LCL_ABL < 3:
            return 1
        else:
            return 4
    
    # Situació ES-IN (baixa inestabilitat)
    if ABLgradient > 0.0035 and 0.003 <= FTgradient < 0.005:
        if da_ABLh < 1500 and LCL_ABL < 3:
            return 2
        else:
            return 4
    
    # Situación ES-ES
    if ABLgradient > 0.0035 and FTgradient > 0.005:
        return 4
    
    # Si ninguna condición anterior se cumple
    return 4

def lcl_bolton(T_k, Td_k):
    T_k  = np.asarray(T_k, dtype=float)
    Td_k = np.asarray(Td_k, dtype=float)

    # Fórmula de Bolton (1980)
    T_lcl = 1.0 / (1.0 / (Td_k - 56.0) + np.log(T_k / Td_k) / 800.0) + 56.0  # K
    z_lcl = (T_k - T_lcl) / 0.0098  # m

    return z_lcl

def z_from_Tq(p_lev_hpa, T_lev, q_lev, p_sfc_hpa, T2, q2):
    p = np.asarray(p_lev_hpa)  # hPa
    T = np.asarray(T_lev)      # K
    q = np.asarray(q_lev)      # kg/kg
    ps = float(p_sfc_hpa)
    T2 = float(T2)
    q2 = float(q2)

    if p[0] < p[-1]:
        p = p[::-1]; T = T[::-1]; q = q[::-1]

    p_full  = np.concatenate(([ps], p))
    Tv_full = np.concatenate(([T2 * (1 + 0.61*q2)], T * (1 + 0.61*q)))

    Rd = 287.05
    g  = 9.80665
    Tv_bar = 0.5 * (Tv_full[:-1] + Tv_full[1:])
    dz = (Rd * Tv_bar / g) * np.log(p_full[:-1] / p_full[1:])
    z_levels = np.cumsum(dz)
    return z_levels * units.meter

# --------------------------------------------------
#           Funció plot                                   
# --------------------------------------------------
def plot_time_mosaic_gridspec(
    data_list,              # llista [time][lat,lon]
    lon, lat,
    times,
    ncols=8,
    cell_w=5.0,
    cell_h=3.9,
    cb_h=0.6,
    cb_pad=0.35,
    top_pad=0.8,
    title="",
    cmap=None,
    norm=None,
    cb_ticks=None,
    cb_ticklabels=None,
    cb_label="",
    invert_cbar=False,
    logo_path=None,
    extent=[0, 3.5, 40.25, 43],
    #extent=[1.25, 8.25, 42, 46],
    #extent=[-2.25, 1, 39.85, 43],
    #extent=[23, 24.5, 37.5, 39.5],
    extra_features=None,
    cbar_mappable=None,
    dpi=300,
):

    nt = len(data_list)
    nrows = int(np.ceil(nt / ncols))

    # ---------- AJUST AUTOMÀTIC DE cell_h SEGONS L'EXTENT ----------
    lon_span = extent[1] - extent[0]
    lat_span = extent[3] - extent[2]
    aspect_extent = lon_span / lat_span      # <- sense cos(mean_lat)
    cell_h = cell_w / aspect_extent

    # ---------- GAP VERTICAL FIX (en polzades, no proporcional) ----------
    hspace_inches = 0.25        # <- ajusta aquest valor al gust (abans equivalia a ~0.97" amb cell_h=3.9)
    hspace_frac = hspace_inches / cell_h

    fig_w = ncols * cell_w
    fig_h = nrows * cell_h + cb_pad + cb_h + top_pad

    fig = plt.figure(figsize=(fig_w, fig_h), dpi=dpi)
    fig.subplots_adjust(top=1 - top_pad / fig_h)

    gs = gridspec.GridSpec(
        nrows=nrows + 2,
        ncols=ncols,
        figure=fig,
        height_ratios=[1] * nrows + [cb_pad / cell_h, cb_h / cell_h],
        hspace=hspace_frac,   # <- en lloc de 0.25 fix
        wspace=0.04
    )

    proj = ccrs.PlateCarree()
    mappable = None

    # ---------- MAPES ----------
    for i in range(nrows * ncols):
        r, c = divmod(i, ncols)
        ax = fig.add_subplot(gs[r, c], projection=proj)

        if i >= nt:
            ax.set_axis_off()
            continue

        ax.set_extent(extent, crs=proj)
        ax.set_autoscale_on(False)
        ax.set_aspect("equal", adjustable="box")

        ax.add_feature(cfeature.OCEAN, facecolor="lightblue", zorder=10)
        ax.add_feature(cfeature.COASTLINE, zorder=4)
        ax.add_feature(cfeature.BORDERS, zorder=5)
        ax.add_feature(cfeature.STATES, zorder=6, linewidth=0.15)

        # features
        if extra_features:
            for feat, style in extra_features:
                ax.add_feature(feat, **style)

        #Z = data_list[i]
        Z = np.asarray(data_list[i], dtype=float)

        m = ax.pcolormesh(
            lon, lat, Z,
            cmap=cmap, norm=norm,
            shading="nearest",
            transform=proj,
            zorder=1
        )
        if mappable is None:
            mappable = m

        ax.set_title(times[i].strftime("%Y-%m-%d %H") + "Z", fontsize=13)
        ax.set_xticks([]); ax.set_yticks([])

    # ---------- GAP ----------
    fig.add_subplot(gs[nrows, :]).axis("off")

    # ---------- COLORBAR (centrada) ----------
    sub = gs[nrows + 1, :].subgridspec(1, 3, width_ratios=[3, 6, 3], wspace=0.0)
    fig.add_subplot(sub[0, 0]).axis("off")

    ax_cb = fig.add_subplot(sub[0, 1])
    ax_cb.axis("off")
    cax = ax_cb.inset_axes([0.10, 0.25, 0.80, 0.50])

    cb_source = cbar_mappable if cbar_mappable is not None else mappable
    cb = fig.colorbar(cb_source, cax=cax, orientation="horizontal")

    if cb_ticks is not None:
        cb.set_ticks(cb_ticks)
    if cb_ticklabels is not None:
        cb.set_ticklabels(cb_ticklabels)
    if invert_cbar:
        cb.ax.invert_xaxis()

    cb.set_label(cb_label, fontsize=12)
    cb.ax.tick_params(labelsize=11)

    fig.add_subplot(sub[0, 2]).axis("off")

    # ---------- LOGO (GAP + COLORBAR, a la dreta) ----------
    if logo_path:
        ax_logo = fig.add_subplot(gs[nrows:nrows + 2, -2:])
        ax_logo.axis("off")
        logo = plt.imread(logo_path)
        ax_logo.imshow(logo, aspect="equal")
        ax_logo.set_anchor("E")

    # TÍTOL (suptitle dins top_pad) igual que AGO
    top_frac = top_pad / fig_h
    fig.suptitle(title, fontsize=22, y=1 - top_frac / 2)

    return fig

# ──────────────────────────────────────────────────────────────────────────
# WARMUP: estat real de 'durations' i 'rain_timer' (→ precip_prev) des de SMC
# ──────────────────────────────────────────────────────────────────────────
def smc_path(station, d):
    return f"{SMC_ROOT}/{d.year:04d}/{d.month:02d}/station={station}/data_{d.isoformat()}.parquet"

def meta_path(d):
    return f"{SMC_ROOT}/metadata/{d.year:04d}/{d.month:02d}/stations_metadata_{d.isoformat()}.parquet"

def stations_available(d):
    pat = f"{SMC_ROOT}/{d.year:04d}/{d.month:02d}/station=*/data_{d.isoformat()}.parquet"
    return sorted(re.search(r"station=([^/]+)/", p).group(1) for p in glob.glob(pat))

def load_stations_metadata(d):
    meta = pd.read_parquet(meta_path(d))
    return meta.set_index("station_code")

def idw_interpolation(x, y, z, xi, yi, power=2, k=6):
    xy_st   = np.vstack([x, y]).T
    xy_grid = np.vstack([xi.ravel(), yi.ravel()]).T
    tree = cKDTree(xy_st)
    dist, idx = tree.query(xy_grid, k=k)
    dist[dist == 0] = 1e-12
    w = 1 / (dist ** power)
    w /= w.sum(axis=1, keepdims=True)
    result = np.sum(z[idx] * w, axis=1)
    return result.reshape(xi.shape)

def load_station_hourly(station, start_date, end_date):
    frames, d = [], start_date
    while d <= end_date:
        p = Path(smc_path(station, d))
        if p.exists():
            frames.append(pd.read_parquet(p))
        d += timedelta(days=1)
    if not frames:
        return None
    df = pd.concat(frames, ignore_index=True)
    df = df[df["state"].isin(["V", "T", " "])]
    wide = df.pivot_table(index="date", columns="variable_code", values="value").sort_index()
    if SMC["temp"] not in wide or SMC["hr"] not in wide or SMC["rad"] not in wide:
        return None

    def getcol(code, fill=None):
        s = wide[code] if code in wide.columns else pd.Series(np.nan, index=wide.index)
        return s if fill is None else s.fillna(fill)

    out = pd.DataFrame(index=wide.index)
    out["temp_C"]    = getcol(SMC["temp"])
    out["hr_pct"]    = getcol(SMC["hr"])
    out["hrmin_pct"] = getcol(SMC["hrmin"])
    out["tmax_C"]    = getcol(SMC["tmax"])
    out["tmin_C"]    = getcol(SMC["tmin"])
    out["precip_mm"] = getcol(SMC["precip"], 0.0)
    out["rad_Wm2"]   = getcol(SMC["rad"], 0.0).clip(lower=0.0)
    h = pd.DataFrame()
    h["temp_C"]    = out["temp_C"].resample("1h").mean()
    h["hr_pct"]    = out["hr_pct"].resample("1h").mean()
    h["hrmin_pct"] = out["hrmin_pct"].resample("1h").min()
    h["tmax_C"]    = out["tmax_C"].resample("1h").max()
    h["tmin_C"]    = out["tmin_C"].resample("1h").min()
    h["precip_mm"] = out["precip_mm"].resample("1h").sum()
    h["rad_Wm2"]   = out["rad_Wm2"].resample("1h").mean()

    h = h.interpolate(method="linear", limit=3)
    return h.dropna(subset=["temp_C", "hr_pct"])

def build_warmup_state(run_start_dt, target_lon2d, target_lat2d,
                        llindar_hr=60, warmup_days_short=5,
                        min_estacions=3):
    """
    target_lon2d, target_lat2d: meshgrid de ds_surf.longitude / ds_surf.latitude
    JA RETALLAT a Catalunya.

    'durations' només es reinicialitza a 0 quan la HR ha estat >= llindar_hr
    durant MÉS de 3 hores consecutives (no a la primera hora per sobre).
    """
    warm_start = (run_start_dt - timedelta(days=warmup_days_short)).date()
    warm_end   = (run_start_dt - timedelta(hours=1)).date()

    all_dates = pd.date_range(warm_start, warm_end, freq="D").date
    estacions = set()
    for d in all_dates:
        estacions.update(stations_available(d))

    series = {}
    for st in sorted(estacions):
        h = load_station_hourly(st, warm_start, warm_end)
        if h is not None and len(h) > 24:
            series[st] = h

    if not series:
        raise RuntimeError("Sense dades d'estacions per al període de warmup.")

    meta = load_stations_metadata(warm_end)

    full_index = pd.date_range(
        pd.Timestamp(warm_start),
        pd.Timestamp(run_start_dt) - timedelta(hours=1),
        freq="1h"
    )

    shape = target_lon2d.shape
    durations  = np.zeros(shape)
    above_thr_hours = np.zeros(shape)   # hores consecutives amb HR >= llindar
    precip_acc = np.zeros(shape)
    rain_timer = np.zeros(shape)
    precip_prev_local = np.zeros(shape)

    for ts in full_index:
        hr_vals, pr_vals, lons, lats = [], [], [], []
        for st, h in series.items():
            if ts in h.index and st in meta.index:
                row = h.loc[ts]
                if not np.isnan(row["hr_pct"]):
                    hr_vals.append(row["hr_pct"])
                    pr_vals.append(0.0 if np.isnan(row["precip_mm"]) else row["precip_mm"])
                    lons.append(float(meta.loc[st, "longitude"]))
                    lats.append(float(meta.loc[st, "latitude"]))

        if len(lons) < min_estacions:
            continue

        lons_arr, lats_arr = np.array(lons), np.array(lats)
        k_eff = min(6, len(lons_arr))

        hr_grid = idw_interpolation(lons_arr, lats_arr, np.array(hr_vals),
                                     target_lon2d, target_lat2d, k=k_eff)
        pr_grid = idw_interpolation(lons_arr, lats_arr, np.array(pr_vals),
                                     target_lon2d, target_lat2d, k=k_eff)

        Ac_HR = hr_grid < llindar_hr

        # Comptador d'hores consecutives per sobre del llindar
        above_thr_hours = np.where(Ac_HR, 0, above_thr_hours + 1)

        # Només reinicialitza durations quan porta MÉS de 3h per sobre del llindar
        reset_mask = above_thr_hours > 3
        durations = np.where(Ac_HR, durations + 1,
                     np.where(reset_mask, 0, durations))

        precip_acc = precip_acc + np.nan_to_num(pr_grid, nan=0.0)
        delta = precip_acc - precip_prev_local
        trigger = (precip_acc > 12) & (delta >= 1)
        rain_timer = np.where(trigger, (precip_acc - 5) / 2, rain_timer)
        rain_timer = np.maximum(rain_timer - 1/8, 0)
        precip_prev_local = precip_acc.copy()

    return durations, rain_timer


# ══════════════════════════════════════════════════════════════════════════
#   VERSIÓ HÍBRIDA (EUROPE_MODE): SMC dins Catalunya, model ICON-EU fora
# ══════════════════════════════════════════════════════════════════════════
def build_warmup_state_hybrid(run_start_dt, target_lon2d, target_lat2d, inside_cat_mask,
                               europe_hr_hourly, europe_pr_hourly, europe_times,
                               llindar_hr=60, warmup_days_short=5, min_estacions=3):
    """
    Igual que build_warmup_state, però híbrida:
      - Dins de Catalunya (inside_cat_mask=True): IDW des d'estacions SMC,
        exactament com sempre (mateixa precisió que abans).
      - Fora de Catalunya: HR/precipitació ja calculades pel propi ICON-EU
        (o l'històric que facis servir), sense estacions.

    Paràmetres nous respecte a l'original:
      inside_cat_mask   : array booleà (mateixa forma que target_lon2d),
                          True = dins de Catalunya.
      europe_hr_hourly  : array (ntime, nlat, nlon) — HR% horària del model,
                          MATEIXA graella que target_lon2d/target_lat2d,
                          cobrint com a mínim el període de warmup
                          (warmup_days_short dies abans del run).
      europe_pr_hourly  : array (ntime, nlat, nlon) — precipitació horària
                          (mm), mateixa graella i període.
      europe_times      : pd.DatetimeIndex (o llista de pd.Timestamp),
                          eix temporal dels dos arrays anteriors.

    TODO: connecta europe_hr_hourly / europe_pr_hourly a la teva font real
    (ICON-EU arxivat de runs anteriors, ERA5-Land horari, etc.) — aquí
    només s'assumeix que ja els tens carregats com a arrays.
    """
    warm_start = (run_start_dt - timedelta(days=warmup_days_short)).date()
    warm_end   = (run_start_dt - timedelta(hours=1)).date()

    all_dates = pd.date_range(warm_start, warm_end, freq="D").date
    estacions = set()
    for d in all_dates:
        estacions.update(stations_available(d))

    series = {}
    for st in sorted(estacions):
        h = load_station_hourly(st, warm_start, warm_end)
        if h is not None and len(h) > 24:
            series[st] = h

    meta = load_stations_metadata(warm_end)

    full_index = pd.date_range(
        pd.Timestamp(warm_start),
        pd.Timestamp(run_start_dt) - timedelta(hours=1),
        freq="1h"
    )

    europe_times = pd.DatetimeIndex(europe_times)

    shape = target_lon2d.shape
    durations  = np.zeros(shape)
    above_thr_hours = np.zeros(shape)
    precip_acc = np.zeros(shape)
    rain_timer = np.zeros(shape)
    precip_prev_local = np.zeros(shape)

    for ts in full_index:
        # ── Catalunya: IDW d'estacions SMC (idèntic a l'original) ──
        hr_vals, pr_vals, lons, lats = [], [], [], []
        for st, h in series.items():
            if ts in h.index and st in meta.index:
                row = h.loc[ts]
                if not np.isnan(row["hr_pct"]):
                    hr_vals.append(row["hr_pct"])
                    pr_vals.append(0.0 if np.isnan(row["precip_mm"]) else row["precip_mm"])
                    lons.append(float(meta.loc[st, "longitude"]))
                    lats.append(float(meta.loc[st, "latitude"]))

        hr_grid_cat = pr_grid_cat = None
        if len(lons) >= min_estacions:
            lons_arr, lats_arr = np.array(lons), np.array(lats)
            k_eff = min(6, len(lons_arr))
            hr_grid_cat = idw_interpolation(lons_arr, lats_arr, np.array(hr_vals),
                                             target_lon2d, target_lat2d, k=k_eff)
            pr_grid_cat = idw_interpolation(lons_arr, lats_arr, np.array(pr_vals),
                                             target_lon2d, target_lat2d, k=k_eff)

        # ── resta d'Europa: camp del model, ja a la graella ──
        if ts in europe_times:
            t_idx = europe_times.get_loc(ts)
            hr_grid_eu = europe_hr_hourly[t_idx]
            pr_grid_eu = europe_pr_hourly[t_idx]
        else:
            hr_grid_eu = np.full(shape, np.nan)
            pr_grid_eu = np.zeros(shape)

        # ── combina: dins CAT -> estacions; fora -> model ──
        if hr_grid_cat is not None:
            hr_grid = np.where(inside_cat_mask, hr_grid_cat, hr_grid_eu)
            pr_grid = np.where(inside_cat_mask, pr_grid_cat, pr_grid_eu)
        else:
            hr_grid, pr_grid = hr_grid_eu, pr_grid_eu

        Ac_HR = hr_grid < llindar_hr
        above_thr_hours = np.where(Ac_HR, 0, above_thr_hours + 1)
        reset_mask = above_thr_hours > 3
        durations = np.where(Ac_HR, durations + 1, np.where(reset_mask, 0, durations))

        precip_acc = precip_acc + np.nan_to_num(pr_grid, nan=0.0)
        delta = precip_acc - precip_prev_local
        trigger = (precip_acc > 12) & (delta >= 1)
        rain_timer = np.where(trigger, (precip_acc - 5) / 2, rain_timer)
        rain_timer = np.maximum(rain_timer - 1/8, 0)
        precip_prev_local = precip_acc.copy()

    return durations, rain_timer

In [ ]:
# ============================================================
#   CONFIGURACIÓ
# ============================================================
date       = pd.Timestamp(ds_surf.time.values[0]).strftime("%Y%m%d")
Llindar_HR = 60   # % humitat relativa

HHL_DIR = Path("/home/labfire/data/GRAF_MAPS/static_data/HHL")   # ← carpeta amb els fitxers HHL i HSURF
RUN     = "2026031200"         # ← run del model

# ── Colormaps ────────────────────────────────────────────────
Finestra_palet     = ['white', 'xkcd:light beige', 'xkcd:yellow', 'gold',
                      'darkorange', 'red', 'darkred']
levels_complexitat = [0, 1, 2, 3, 4, 5, 6, 7]
cmap_complexitat   = colors.ListedColormap(Finestra_palet)
norm_complexitat   = BoundaryNorm(levels_complexitat,
                                  ncolors=cmap_complexitat.N, clip=True)

colors_list          = ['darkred', 'gold', 'yellow', 'lightgreen']
levels_piroconveccio = [0.5, 1.5, 2.5, 3.5, 4.5]
colors_grad          = colors.ListedColormap(colors_list)
norm_piroconveccio   = BoundaryNorm(levels_piroconveccio,
                                    ncolors=colors_grad.N, clip=True)

# ── Dimensions ───────────────────────────────────────────────
new_lat        = ds_surf.latitude
new_lon        = ds_surf.longitude
num_time_steps = len(ds_surf.time)

# ── Acumuladors temporals ────────────────────────────────────
# durations   = np.ones((len(new_lat), len(new_lon)))
# precip_prev = np.zeros((len(new_lat), len(new_lon)))
# rain_timer  = np.zeros((len(new_lat), len(new_lon)))
new_lat        = ds_surf.latitude
new_lon        = ds_surf.longitude
num_time_steps = len(ds_surf.time)

lon2d_fc, lat2d_fc = np.meshgrid(new_lon.values, new_lat.values)

run_start_dt = pd.Timestamp(ds_surf.time.values[0]).to_pydatetime()

def load_europe_warmup_from_icon_archive(run_start_dt, warmup_days=5,
                                          archive_dir="/home/labfire/data/ICON-EU"):
    """
    Reconstrueix HR% i precipitació horàries per tot Europa des de
    `warmup_days` dies enrere fins just abans del run actual.

    Per als dies COMPLETS anteriors, agafa les 24h senceres del run de
    les 00Z arxivat d'aquell dia. Per AVUI (si el run actual és a les
    12Z o més tard), agafa nomès les hores 0..RUN_HOUR-1 del run de les
    00Z d'avui — sense això, un run de les 12Z es quedava sense les
    hores 00-11 d'avui mateix (bug: només s'iterava sobre dies passats
    complets, mai el dia del run en si).

    Assumeix fitxers ICONEU_surface_{YYYYMMDD}_00.nc.
    """
    hr_list, pr_list, time_list = [], [], []

    def _load_day(day, hours_slice):
        tag = day.strftime("%Y%m%d") + "_00"
        path = f"{archive_dir}/ICONEU_surface_{tag}.nc"
        if not Path(path).exists():
            print(f"⚠ Falta {path}, es salta aquest dia (warmup incomplet)")
            return

        ds_day = xr.open_dataset(path).isel(time=hours_slice)

        # transpose explícit perquè l'eix "time" sempre sigui el primer —
        # evita que xr.concat/diff canviïn l'ordre dels eixos en acabar
        # convertint a numpy (era la causa del ValueError de shapes).
        hr_vals = ds_day["relhum_2m"].transpose("time", ...).values
        precip_accum_vals = ds_day["tot_prec"].transpose("time", ...).values

        # desacumula: cada hora - l'anterior (la primera es queda tal qual)
        precip_hourly_vals = np.diff(precip_accum_vals, axis=0,
                                      prepend=precip_accum_vals[0:1])
        precip_hourly_vals = np.clip(precip_hourly_vals, 0, None)  # per si hi ha reset del comptador

        hr_list.append(hr_vals)
        pr_list.append(precip_hourly_vals)
        time_list.extend(pd.to_datetime(ds_day.time.values))

        ds_day.close()

    # dies COMPLETS anteriors (com sempre)
    for d in range(warmup_days, 0, -1):
        day = (run_start_dt - timedelta(days=d)).date()
        _load_day(day, slice(0, 24))

    # AVUI, nomès si el run actual és més tard que les 00Z (p.ex. 12Z) —
    # agafa hores 0..RUN_HOUR-1 del run de les 00Z d'avui.
    if run_start_dt.hour > 0:
        _load_day(run_start_dt.date(), slice(0, run_start_dt.hour))

    if not hr_list:
        raise RuntimeError("Cap fitxer d'arxiu ICON-EU trobat pel període de warmup.")

    europe_hr_hourly = np.concatenate(hr_list, axis=0)
    europe_pr_hourly = np.concatenate(pr_list, axis=0)
    europe_times = pd.DatetimeIndex(time_list)

    return europe_hr_hourly, europe_pr_hourly, europe_times


if EUROPE_MODE:
    # ── Versió híbrida: SMC dins Catalunya, model ICON-EU per la resta ──
    import sys
    sys.path.insert(0, os.path.join(ONOFRE_REPO, "generator"))  # ajusta si cal
    from generate_frames import build_polygon_mask
    inside_cat_mask = build_polygon_mask(new_lon.values, new_lat.values, comarques.geometry)

    europe_hr_hourly, europe_pr_hourly, europe_times = load_europe_warmup_from_icon_archive(
        run_start_dt=run_start_dt,
        warmup_days=5,
    )

    durations, rain_timer = build_warmup_state_hybrid(
        run_start_dt=run_start_dt,
        target_lon2d=lon2d_fc,
        target_lat2d=lat2d_fc,
        inside_cat_mask=inside_cat_mask,
        europe_hr_hourly=europe_hr_hourly,
        europe_pr_hourly=europe_pr_hourly,
        europe_times=europe_times,
        llindar_hr=Llindar_HR,
        warmup_days_short=5,
    )
else:
    durations, rain_timer = build_warmup_state(
        run_start_dt=run_start_dt,
        target_lon2d=lon2d_fc,
        target_lat2d=lat2d_fc,
        llindar_hr=Llindar_HR,
        warmup_days_short=5,
    )

precip_prev = np.zeros(lon2d_fc.shape)


Finestra_IF_list         = []
Tipus_Piroconveccio_list = []
times                    = []

# ── DC percentils (estàtics, calculats una sola vegada) ──────
dc_vals = ds_DC_cat['DC'].values

# Si el DC ha fallat per a (gairebé) tot el domini (p.ex. GWIS caigut avui,
# vegeu descarrega_dc_effis), NO forcem cap_dc=0 arreu — això deixaria el
# mapa de complexitat tot a 0 en silenci, sense cap error, com si no hi
# hagués risc enlloc. En comptes d'això, s'omet la restricció del DC
# aquell dia (cap_dc sense limitar) i s'avisa ben fort.
frac_nan_dc = float(np.isnan(dc_vals).mean())
if frac_nan_dc > 0.95:
    print(f"⚠⚠⚠ DC no disponible per a ~tot el domini ({frac_nan_dc*100:.0f}% NaN) — "
          f"probablement ha fallat GWIS avui. S'OMET la restricció per DC "
          f"(cap_dc sense limitar, bonus_dc=0) en comptes de forçar complexitat=0 arreu.")
    cap_dc   = np.full(dc_vals.shape, 6)
    bonus_dc = np.zeros(dc_vals.shape)
else:
    p10_dc = ds_perc_cat['DC_quantile'].sel(quantile=0.10).values
    p25_dc = ds_perc_cat['DC_quantile'].sel(quantile=0.25).values
    p50_dc = ds_perc_cat['DC_quantile'].sel(quantile=0.50).values
    p75_dc = ds_perc_cat['DC_quantile'].sel(quantile=0.75).values
    p90_dc = ds_perc_cat['DC_quantile'].sel(quantile=0.90).values

    # cap_dc = np.where(np.isnan(dc_vals), 0,
    #          np.where(dc_vals < p10_dc, 0,
    #          np.where(dc_vals < p25_dc, 2,
    #          np.where(dc_vals < p50_dc, 2,
    #          np.where(dc_vals < p75_dc, 3, 6)))))
    cap_dc = np.where(np.isnan(dc_vals), 0,
             np.where(dc_vals < p10_dc, 0,
             np.where(dc_vals < p25_dc, 2, 6)))
    bonus_dc = np.where(dc_vals >= p90_dc, 1, 0)


def to_scalar(x):
    mag = getattr(x, 'm', getattr(x, 'magnitude', x))
    arr = np.asarray(mag)
    return float(arr.reshape(-1)[0])


# ============================================================
#   PRE-CÀLCUL BLH: geometria estàtica (fora del bucle)
#   z_mc_crop i hsurf_crop no canvien amb el temps
# ============================================================
print("Carregant geometria del model (HHL/HSURF)...")

# --- HSURF ---
hsurf_file = next(HHL_DIR.glob("*HSURF*.grib2"))
ds_hsurf   = cfgrib.open_dataset(str(hsurf_file))
hsurf_var  = list(ds_hsurf.data_vars)[0]
hsurf      = ds_hsurf[hsurf_var]

# --- HHL → z_mc (alçades dels centres dels nivells sobre MSL) ---
hhl_dict = {}
for f in sorted(HHL_DIR.glob(f"*{RUN}*HHL*.grib2")):
    parts = f.stem.split("_")
    lev   = int(parts[-2])
    ds_tmp = cfgrib.open_dataset(str(f))
    var    = list(ds_tmp.data_vars)[0]
    hhl_dict[lev] = ds_tmp[var].values

nivells_ifc  = sorted(hhl_dict.keys())   # 47–75
HHL_arr      = np.stack([hhl_dict[k] for k in nivells_ifc], axis=0)
model_levels = list(range(25, 75))   # 50 nivells
ifc_base     = 25

z_mc_list = []
for ml in model_levels:
    idx_top = ml - ifc_base
    idx_bot = ml - ifc_base + 1
    z_mc_list.append((HHL_arr[idx_top] + HHL_arr[idx_bot]) / 2.0)
z_mc = np.stack(z_mc_list, axis=0)   # (27, 657, 1377)

# --- Retall a Catalunya (igual que ds_surf / ds_per) ---
lats_full = hsurf.latitude.values
lons_full = hsurf.longitude.values
lat_min   = float(ds_surf.latitude.min())
lat_max   = float(ds_surf.latitude.max())
lon_min   = float(ds_surf.longitude.min())
lon_max   = float(ds_surf.longitude.max())

lat_idx = np.where((lats_full >= lat_min - 0.01) & (lats_full <= lat_max + 0.01))[0]
lon_idx = np.where((lons_full >= lon_min - 0.01) & (lons_full <= lon_max + 0.01))[0]

hsurf_crop = hsurf.values[np.ix_(lat_idx, lon_idx)]                              # (44, 57)
z_mc_crop  = z_mc[:, np.ix_(lat_idx, lon_idx)[0], np.ix_(lat_idx, lon_idx)[1]]  # (27, 44, 57)

print(f"  hsurf_crop: {hsurf_crop.shape}  z_mc_crop: {z_mc_crop.shape}  ✅")


# ── Funcions BLH — rèplica exacta del codi dels companys ────
# Referència: nivell 72 (~90m AGL)  ≡  índex 10 de la sonda
# Fricció:    nivell 73 (~40m AGL)  ≡  vv_40 dels companys
# Ri_crit:    0.33
 
g_blh       = 9.81
Ri_crit_blh = 0.33
BLH_MIN     = 50.0
BLH_MAX     = 3000.0
 
def _calc_theta_v(T, p, qv):
    p0    = 100000.0
    R_d   = 287.05
    c_pd  = 1004.0
    kappa = R_d / c_pd
    return T * (p0 / p)**kappa * (1.0 + 0.608 * qv)
 
def _friction(ws_40m):
    """
    Velocitat de fricció a partir del vent a ~40m AGL.
    Replica: f = 0.4 * vv_40 * (1/ln(100))
    Els companys usen ln(100) perquè la seva referència és ~100m.
    """
    return 0.4 * ws_40m * (1.0 / np.log(100.0))
 
def _rib(tpv0, tpvi, h0, hi, u0, ui, v0, vi, f):
    """
    Bulk Richardson Number — Vogelezang & Holtslag (1996).
    Rèplica exacta de rib_new() del codi dels companys.
    """
    num = (g_blh / tpv0) * (tpvi - tpv0) * (hi - h0)
    den = (ui - u0)**2 + (vi - v0)**2 + 100.0 * f**2
    den = np.maximum(den, 1e-6)
    return num / den
 
def calc_BLH(per_t, z_mc_crop, hsurf_crop):
    """
    Calcula BLH (m AGL) replicant exactament el mètode dels companys.
 
    per_t      : ds_per.isel(time=t) ordenat model_level descendent
                 índex 0 = nivell 74 (~10m AGL)
                 índex 1 = nivell 73 (~40m AGL)  ← fricció
                 índex 2 = nivell 72 (~90m AGL)  ← referència
                 índex 3 = nivell 71 (~156m AGL) ← primer nivell calculat
                 ...
    z_mc_crop  : (27, nlat, nlon) MSL — ha d'estar invertit igual que per_t
    hsurf_crop : (nlat, nlon) orografia MSL
    """
    T  = per_t['t'].values
    qv = per_t['qv'].values
    p  = per_t['p'].values
    u  = per_t['u'].values
    v  = per_t['v'].values
 
    tpv = _calc_theta_v(T, p, qv)   # (27, nlat, nlon)
 
    # Alçades AGL — z_mc_crop invertit per alinear amb per_t_sorted
    z_mc_inv = z_mc_crop[::-1, :, :]
    z_agl    = np.maximum(z_mc_inv - hsurf_crop[np.newaxis, :, :], 0.0)
 
    # Índexs fixos (equivalents al codi dels companys)
    idx_40m  = 1   # nivell 73, ~40m AGL  → fricció (vv_40)
    idx_ref  = 2   # nivell 72, ~90m AGL  → referència (índex 10 companys)
 
    # Fricció a partir del vent a ~40m AGL
    ws_40m = np.sqrt(u[idx_40m]**2 + v[idx_40m]**2)
    f_fric = _friction(ws_40m)
 
    # Referència
    tpv0 = tpv[idx_ref]
    u0   = u[idx_ref]
    v0   = v[idx_ref]
    h0   = z_agl[idx_ref]
 
    # Ri_b per a tots els nivells per sobre de la referència
    nlev = T.shape[0]
    rib  = np.full_like(T, np.nan)
    for k in range(idx_ref + 1, nlev):
        rib[k] = _rib(tpv0, tpv[k], h0, z_agl[k],
                      u0, u[k], v0, v[k], f_fric)
 
    # BLH = primer nivell on Ri_b >= 0.33 pujant des de la referència
    # (replica: for j in Rib_new[10:]: if j >= 0.33: BLH = z[...]; break)
    BLH = np.zeros(tpv0.shape, dtype=np.float32)
    for k in range(idx_ref + 1, nlev):
        mask = (rib[k] >= Ri_crit_blh) & (BLH == 0)
        BLH  = np.where(mask, z_agl[k], BLH)
 
    BLH = np.clip(BLH, BLH_MIN, BLH_MAX)
    return BLH

# ============================================================
#   BUCLE TEMPORAL
# ============================================================
for t in range(num_time_steps):

    t_val  = pd.to_datetime(ds_surf.time.values[t])
    surf_t = ds_surf.isel(time=t)
    per_t  = ds_per.isel(time=t)

    # ── Variables de superfície ──────────────────────────────
    da_temp2m   = surf_t['t_2m']    * units.kelvin
    da_DewP2m   = surf_t['td_2m']   * units.kelvin
    da_HR       = surf_t['relhum_2m']
    da_Ucomp    = surf_t['u_10m']   * units('m/s')
    da_Vcomp    = surf_t['v_10m']   * units('m/s')
    da_Ac_Pluja = surf_t['tot_prec']

    # ── ABLh: càlcul propi (Vogelezang-Holtslag) ─────────────
    # Ordenem ds_per de baix a dalt (model_level descendent)
    per_t_sorted = per_t.sortby("model_level", ascending=False)

    BLH_VH   = calc_BLH(per_t_sorted, z_mc_crop, hsurf_crop)
    ABLh_arr = np.maximum(BLH_VH, 50.0)

    # Si vols conservar el MH del model com a referència per comparar:
    # da_ABLh_model = surf_t['mh']

    da_ABLh = xr.DataArray(
        ABLh_arr,
        dims=["latitude", "longitude"],
        coords={"latitude": new_lat, "longitude": new_lon}
    )

    # ── Perfil de temperatura (baix→amunt) ───────────────────
    da_temp      = per_t_sorted['t'] * units.kelvin
    da_p_per     = per_t_sorted['p'] / 100.0             # Pa → hPa

    ds_surface_press = da_p_per.isel(model_level=0) * units.hPa

    # ── Vent ─────────────────────────────────────────────────
    wind_speed = np.sqrt(da_Ucomp**2 + da_Vcomp**2).values * 3.6   # km/h

    # ── Temperatura de rosada en °C ──────────────────────────
    da_DewP2m_celsius = (da_DewP2m.metpy.convert_units("degC")
                                   .metpy.dequantify().values)

    # ── Acumulació hores HR < llindar ────────────────────────
    Ac_HR     = da_HR.values < Llindar_HR
    durations = np.where(Ac_HR, durations + 3, 0)

    # ── Mida AGO ─────────────────────────────────────────────
    wind_run = np.zeros_like(wind_speed)
    wind_run = np.where(durations < 6,                       wind_speed * 0.025, wind_run)
    wind_run = np.where((durations >= 6) & (durations < 10), wind_speed * 0.05,  wind_run)
    wind_run = np.where(durations >= 10,                     wind_speed * 0.1,   wind_run)

    # ── Gradients tèrmics (VECTORITZAT — sense bucles per píxel) ──────
    # Abans: for i in range(nlat): for j in range(nlon): ... (inviable a
    # escala Europa: ~900.000 píxels x N timesteps). Ara: numpy vectoritzat
    # sobre tota la graella d'un cop, mateixa fórmula hipsomètrica i mateixa
    # interpolació lineal que abans, però sense bucle Python per píxel.
    T_vals   = da_temp.values
    p_vals   = da_p_per.values
    qv_vals  = per_t_sorted["qv"].values
    T2m_arr  = da_temp2m.values
    qv2_arr  = per_t_sorted["qv"].isel(model_level=0).values
    p_sf_arr = da_p_per.isel(model_level=0).values

    nlat, nlon = T2m_arr.shape
    nl = T_vals.shape[0]

    Rd, g = 287.05, 9.80665
    p_full  = np.concatenate([p_sf_arr[None, :, :], p_vals], axis=0)
    Tv_full = np.concatenate(
        [(T2m_arr * (1 + 0.61 * qv2_arr))[None, :, :],
         T_vals * (1 + 0.61 * qv_vals)], axis=0
    )
    Tv_bar = 0.5 * (Tv_full[:-1] + Tv_full[1:])
    dz     = (Rd * Tv_bar / g) * np.log(p_full[:-1] / p_full[1:])
    z_3d   = np.cumsum(dz, axis=0)  # (nl, nlat, nlon) — mateix z_3d d'abans

    def interp_along_levels_vec(y_3d, z_3d, z_target_2d):
        """
        Equivalent vectoritzat de fer, per cada píxel (i,j):
            np.interp(z_target_2d[i,j], z_3d[:,i,j], y_3d[:,i,j])
        però per a tota la graella en un sol pas (sense bucle Python).
        """
        nl_ = z_3d.shape[0]
        below = np.clip((z_3d <= z_target_2d[None, :, :]).sum(axis=0) - 1, 0, nl_ - 2)
        above = below + 1
        iy, ix = np.indices((nlat, nlon))

        z_lo, z_hi = z_3d[below, iy, ix], z_3d[above, iy, ix]
        y_lo, y_hi = y_3d[below, iy, ix], y_3d[above, iy, ix]

        denom = np.where((z_hi - z_lo) == 0, np.nan, z_hi - z_lo)
        frac  = (z_target_2d - z_lo) / denom
        result = y_lo + frac * (y_hi - y_lo)

        # clip als extrems (equivalent a left=/right= de np.interp)
        result = np.where(z_target_2d <= z_3d[0],  y_3d[0],  result)
        result = np.where(z_target_2d >= z_3d[-1], y_3d[-1], result)
        return result

    T_at_abl     = interp_along_levels_vec(T_vals, z_3d, ABLh_arr)
    T_at_abl2000 = interp_along_levels_vec(T_vals, z_3d, ABLh_arr + 2000)
    p_at_abl     = interp_along_levels_vec(p_vals, z_3d, ABLh_arr)
    p_at_abl2000 = interp_along_levels_vec(p_vals, z_3d, ABLh_arr + 2000)

    p_abl_xr = xr.DataArray(
        p_at_abl, dims=["latitude", "longitude"],
        coords={"latitude": new_lat, "longitude": new_lon}
    ) * units("hPa")

    p_abl2000_xr = xr.DataArray(
        p_at_abl2000, dims=["latitude", "longitude"],
        coords={"latitude": new_lat, "longitude": new_lon}
    ) * units("hPa")

    Theta_ABL     = mpcalc.potential_temperature(p_abl_xr,     T_at_abl     * units.kelvin)
    Theta_ABL2000 = mpcalc.potential_temperature(p_abl2000_xr, T_at_abl2000 * units.kelvin)

    gradient_surface_to_abl = (Theta_ABL - da_temp2m) / (ABLh_arr * units.meter)
    gradient_abl_to_2000m   = (Theta_ABL2000 - Theta_ABL) / (2000 * units.meter)

    gradients = xr.apply_ufunc(
        calcul_gradients,
        gradient_surface_to_abl, gradient_abl_to_2000m,
        vectorize=True)
    gradients = xr.where(
        (da_ABLh >= 2000) & (gradients != 1),
        4, gradients)

    # ── Bloqueig per pluja ───────────────────────────────────
    precip        = da_Ac_Pluja.values
    delta_precip  = precip - precip_prev
    trigger_mask  = (precip > 12) & (delta_precip >= 1)
    rain_timer    = np.where(trigger_mask, (precip - 5) / 2, rain_timer)
    rain_timer    = np.maximum(rain_timer - 1/8, 0)
    precip_prev   = precip.copy()
    bloqueig_mask = rain_timer > 0

    # ── Finestra IF ──────────────────────────────────────────
    Finestra_IF = calculate_Finestra_IF(
        wind_speed, durations, da_DewP2m_celsius,
        wind_run, gradients, da_Ac_Pluja)
    Finestra_IF = np.where(bloqueig_mask, 0, Finestra_IF)

    complexitat = finestra_reclass(Finestra_IF)

    # ── Modulació per DC ──────────────────────────────────────
    complexitat = np.minimum(complexitat, cap_dc)
    #complexitat = np.clip(complexitat + bonus_dc, 0, 6)

    Finestra_IF_list.append(complexitat.copy())

    # ── LCL Bolton (1980) ────────────────────────────────────
    zLCL = lcl_bolton(da_temp2m.values, da_DewP2m.values)
    zLCL_da = xr.DataArray(
        zLCL, dims=["latitude", "longitude"],
        coords={"latitude": new_lat, "longitude": new_lon}
    )
    times.append(t_val)

    # ── Tipus piroconvecció ──────────────────────────────────
    LCL_ABL = zLCL_da / da_ABLh

    Tipus_Piroconveccio1 = xr.apply_ufunc(
        tipus_piroconveccio,
        gradient_surface_to_abl, gradient_abl_to_2000m,
        da_ABLh, LCL_ABL,
        vectorize=True)
    Tipus_Piroconveccio1 = xr.where(complexitat > 2, Tipus_Piroconveccio1, np.nan)
    Tipus_Piroconveccio_list.append(Tipus_Piroconveccio1.copy())


# ============================================================
#   PLOT COMPLEXITAT
# ============================================================
fig = plot_time_mosaic_gridspec(
    data_list=Finestra_IF_list,
    lon=new_lon,
    lat=new_lat,
    times=times,
    cmap=cmap_complexitat,
    norm=norm_complexitat,
    cbar_mappable=None,
    cb_ticks=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5],
    cb_ticklabels=['0', '1', '2', '3', '4', '5', '6'],
    cb_label="Nivell de complexitat",
    title="Nivells de complexitat d'extinció - ICON-EU Forecast",
    logo_path=f"{ONOFRE_REF}/logo_bombers.png",
    extra_features=[
        (regions_feat,   dict(edgecolor="black", linewidth=0.40, zorder=4)),
        (comarques_feat, dict(edgecolor="black", linewidth=0.07, zorder=5)),
        (invCAT_feat,    dict(edgecolor='none', facecolor='white', zorder=3)),
    ]
)
plt.savefig(os.path.join(OUTPUT_PLOTS, f"Complexitat_{date}.png"), dpi=300, bbox_inches="tight", pad_inches=0.12)
plt.show()

# ============================================================
#   PLOT PIROCONVECCIÓ
# ============================================================
cmap_cb = colors.ListedColormap(colors_grad.colors[::-1])
sm_cb   = ScalarMappable(norm=norm_piroconveccio, cmap=cmap_cb)
sm_cb.set_array([])

fig = plot_time_mosaic_gridspec(
    data_list=Tipus_Piroconveccio_list,
    lon=new_lon,
    lat=new_lat,
    times=times,
    cmap=colors_grad,
    norm=norm_piroconveccio,
    cbar_mappable=sm_cb,
    cb_ticks=[1, 2, 3, 4],
    cb_ticklabels=[
        'Convective plumes',
        'Overshooting PyroCu',
        'Resilient PyroCu',
        'PyroCb'
    ],
    cb_label="Tipus de piroconvecció",
    invert_cbar=False,
    title="Tipus de piroconvecció - ICON-EU Forecast",
    logo_path=f"{ONOFRE_REF}/logo_bombers.png",
    extra_features=[
        (regions_feat,   dict(edgecolor="black", linewidth=0.40, zorder=4)),
        (comarques_feat, dict(edgecolor="black", linewidth=0.07, zorder=5)),
        (invCAT_feat,    dict(edgecolor='none', facecolor='white', zorder=3)),
    ]
)
plt.savefig(os.path.join(OUTPUT_PLOTS, f"Piroconveccio_{date}.png"), dpi=300, bbox_inches="tight", pad_inches=0.12)
plt.show()

In [ ]:
# ============================================================
#   EXPORTA FRAMES PEL VISOR WEB (complexitat + piroconvecció)
#   → alimenta onofre-viewer (docs/index.html), reutilitzant
#     exactament els mateixos objectes que ja fas servir pel mosaic.
#   NOTA: aquest pipeline retalla a Catalunya a la cel·la 2 (abans de
#   calcular res), així que aquestes 2 variables són CAT-only ara mateix,
#   no Europa. Emmasquem per costa (Natural Earth, no comarques) perquè
#   la màscara sigui vàlida igual si algun dia s'amplia a tot Europa.
# ============================================================
import sys
sys.path.insert(0, os.path.join(ONOFRE_REPO, "generator"))  # ajusta si cal
from generate_frames import generate_variable, build_land_mask, apply_polygon_mask

run_tag = pd.Timestamp(ds_surf.time.values[0]).strftime("%Y%m%d_%H")

# resolution="10m" és un bon compromís detall/velocitat; "10m" si vols
# costa molt fina i no t'importa que trigui més a rasteritzar.
inside_land = build_land_mask(new_lon.values, new_lat.values, resolution="10m")

extra_features_cat = [
    (regions_feat,   dict(edgecolor="black", linewidth=0.40, zorder=4)),
    (comarques_feat, dict(edgecolor="black", linewidth=0.07, zorder=5)),
    (invCAT_feat,    dict(edgecolor="none", facecolor="white", zorder=3)),
]

Finestra_IF_masked = apply_polygon_mask(Finestra_IF_list, inside_land)

generate_variable(
    "complexitat", Finestra_IF_masked, new_lon.values, new_lat.values, times, run_tag,
    mosaic_fn=plot_time_mosaic_gridspec,
    mosaic_kwargs=dict(
        cbar_mappable=None,
        cb_ticks=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5],
        cb_ticklabels=['0', '1', '2', '3', '4', '5', '6'],
        cb_label="Nivell de complexitat",
        logo_path=f"{ONOFRE_REF}/logo_bombers.png",
        extra_features=extra_features_cat,
    ),
)

# Mateix colormap invertit que ja fas servir per la colorbar de piroconvecció
cmap_cb = colors.ListedColormap(colors_grad.colors[::-1])
sm_cb   = ScalarMappable(norm=norm_piroconveccio, cmap=cmap_cb)
sm_cb.set_array([])

Tipus_Piroconveccio_masked = apply_polygon_mask(Tipus_Piroconveccio_list, inside_land)

generate_variable(
    "piroconveccio", Tipus_Piroconveccio_masked, new_lon.values, new_lat.values, times, run_tag,
    mosaic_fn=plot_time_mosaic_gridspec,
    mosaic_kwargs=dict(
        cbar_mappable=sm_cb,
        cb_ticks=[1, 2, 3, 4],
        cb_ticklabels=['Convective plumes', 'Overshooting PyroCu', 'Resilient PyroCu', 'PyroCb'],
        cb_label="Tipus de piroconvecció",
        invert_cbar=False,
        logo_path=f"{ONOFRE_REF}/logo_bombers.png",
        extra_features=extra_features_cat,
    ),
)

print(f"✓ Frames exportats pel visor web (run {run_tag}): complexitat + piroconvecció")


In [ ]:
# ============================================================
#   DEBUG TEMPORAL — comprova la màscara de terra (esborra després)
# ============================================================
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 8))
plt.pcolormesh(new_lon.values, new_lat.values, inside_land.astype(int), cmap="Greys", shading="nearest")
plt.title("Màscara terra (all_touched=True)")
plt.gca().set_aspect("equal")
plt.savefig("/tmp/check_mask2.png", dpi=150, bbox_inches="tight")
print("Guardat /tmp/check_mask2.png")
print("Píxels marcats com a terra:", inside_land.sum(), "de", inside_land.size)


In [ ]:
"""
ONOFRE — Humitat de combustible d'UN DIA per a totes les estacions SMC → NetCDF ICON-EU
========================================================================================
Calcula mc1/mc10/mc100/mc1000 (Nelson) i LFMC herbaci/llenyós (GSI) per a TOTES
les estacions amb dades el dia objectiu. Guarda el valor de les 23 UTC, interpola
per IDW a la graella ICON-EU (0.0625°), retalla per Catalunya i escriu un NetCDF.
"""

import glob, re
from datetime import date, timedelta
from pathlib import Path
from datetime import date, timedelta

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from shapely.geometry import Polygon
from scipy.spatial import cKDTree

import nfdrs4

# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURACIÓ
# ──────────────────────────────────────────────────────────────────────────────
TARGET_DATE = date.today() - timedelta(days=1)
WARMUP_DAYS  = 60
DATA_START   = date(2026, 1, 1)
SMC_ROOT     = "/home/labfire/data/SMC-STATIONS"
COMARQUES    = f"{ONOFRE_REF}/comarques/comarques_cat.shp"

TARGET_HOUR  = 23

SMC = dict(temp=32, hr=33, precip=35, rad=36, tmax=40, tmin=42, hrmin=44)

OUTPUT_PLOTS = ONOFRE_MAPES

# ──────────────────────────────────────────────────────────────────────────────
# GRID ICON-EU + MÀSCARA CATALUNYA
# ──────────────────────────────────────────────────────────────────────────────
LONS_ICON = np.arange(0, 3.5, 0.0625)
LATS_ICON = np.arange(40.375, 43, 0.0625)
lon_grid, lat_grid = np.meshgrid(LONS_ICON, LATS_ICON)

shp = "/home/labfire/data/GRAF_MAPS/static_data/comarques/comarques_cat.shp"
comarques = gpd.read_file(shp).to_crs(4326)
cat_poly = comarques.union_all()

mask = np.zeros(lon_grid.shape, dtype=bool)
for i in range(lon_grid.shape[0]):
    for j in range(lon_grid.shape[1]):
        cell = Polygon([
            (lon_grid[i,j] - 0.0625/2, lat_grid[i,j] - 0.0625/2),
            (lon_grid[i,j] + 0.0625/2, lat_grid[i,j] - 0.0625/2),
            (lon_grid[i,j] + 0.0625/2, lat_grid[i,j] + 0.0625/2),
            (lon_grid[i,j] - 0.0625/2, lat_grid[i,j] + 0.0625/2),
        ])
        mask[i,j] = cat_poly.intersects(cell)

# ──────────────────────────────────────────────────────────────────────────────
# RUTES I CÀRREGA SMC (format llarg, semihorari -> horari)
# ──────────────────────────────────────────────────────────────────────────────
def smc_path(station, d):
    return f"{SMC_ROOT}/{d.year:04d}/{d.month:02d}/station={station}/data_{d.isoformat()}.parquet"

def meta_path(d):
    return f"{SMC_ROOT}/metadata/{d.year:04d}/{d.month:02d}/stations_metadata_{d.isoformat()}.parquet"

def stations_available(d):
    pat = f"{SMC_ROOT}/{d.year:04d}/{d.month:02d}/station=*/data_{d.isoformat()}.parquet"
    return sorted(re.search(r"station=([^/]+)/", p).group(1) for p in glob.glob(pat))

def load_station_hourly(station, start_date, end_date):
    frames, d = [], start_date
    while d <= end_date:
        p = Path(smc_path(station, d))
        if p.exists():
            frames.append(pd.read_parquet(p))
        d += timedelta(days=1)
    if not frames:
        return None
    df = pd.concat(frames, ignore_index=True)
    df = df[df["state"].isin(["V", "T", " "])]
    wide = df.pivot_table(index="date", columns="variable_code", values="value").sort_index()
    if SMC["temp"] not in wide or SMC["hr"] not in wide or SMC["rad"] not in wide:
        return None

    def getcol(code, fill=None):
        s = wide[code] if code in wide.columns else pd.Series(np.nan, index=wide.index)
        return s if fill is None else s.fillna(fill)

    out = pd.DataFrame(index=wide.index)
    out["temp_C"]    = getcol(SMC["temp"])
    out["hr_pct"]    = getcol(SMC["hr"])
    out["hrmin_pct"] = getcol(SMC["hrmin"])
    out["tmax_C"]    = getcol(SMC["tmax"])
    out["tmin_C"]    = getcol(SMC["tmin"])
    out["precip_mm"] = getcol(SMC["precip"], 0.0)
    out["rad_Wm2"]   = getcol(SMC["rad"], 0.0).clip(lower=0.0)
    h = pd.DataFrame()
    h["temp_C"]    = out["temp_C"].resample("1h").mean()
    h["hr_pct"]    = out["hr_pct"].resample("1h").mean()
    h["hrmin_pct"] = out["hrmin_pct"].resample("1h").min()
    h["tmax_C"]    = out["tmax_C"].resample("1h").max()
    h["tmin_C"]    = out["tmin_C"].resample("1h").min()
    h["precip_mm"] = out["precip_mm"].resample("1h").sum()
    h["rad_Wm2"]   = out["rad_Wm2"].resample("1h").mean()

    h = h.interpolate(method="linear", limit=3)
    return h.dropna(subset=["temp_C", "hr_pct"])

# ──────────────────────────────────────────────────────────────────────────────
# CÀLCUL NFDRS4 PER ESTACIÓ (valor a les 23 UTC del dia objectiu)
# ──────────────────────────────────────────────────────────────────────────────
def c_to_f(t): return t * 9 / 5 + 32

def compute_station(hourly, lat, target_date):
    dead = {}
    for cls, init in [("mc1","initDeadFuelMoisture1"), ("mc10","initDeadFuelMoisture10"),
                      ("mc100","initDeadFuelMoisture100"), ("mc1000","initDeadFuelMoisture1000")]:
        dfm = nfdrs4.DeadFuelMoisture(); getattr(dfm, init)(); dead[cls] = dfm

    herb = nfdrs4.LiveFuelMoisture()
    herb.Initialize(lat, True, True)
    herb.SetUseVPDAvg(False)

    woody = nfdrs4.LiveFuelMoisture()
    woody.Initialize(lat, False, False)
    woody.SetUseVPDAvg(False)

        # ── HERB ──
    herb.SetLimits(
        -2.0, 5.0,       # Tmin (°C): OK per defecte
        1200.0, 5500.0,   # VPD (Pa): pujar ambdós per a Mediterrani
        #1100, 5000,
        36000.0, 39600.0, # Daylength (s): 10-11h, OK
        0.5, 1.5          # Precip ratio: OK
    )
    herb.SetLFMParameters(1.0, 0.5, 40, 200)

    # ── WOODY (escleròfil, encara més tolerant) ──
    woody.SetLimits(
        -2.0, 5.0,
        1500.0, 6500.0,   # Q.ilex manté conductància a VPD > 5 kPa
        36000.0, 39600.0,
        0.5, 1.5
    )
    woody.SetLFMParameters(1.0, 0.5, 50, 170)

    rain_cm, last_day = 0.0, None
    res = {}
    for ts, row in hourly.iterrows():
        rain_cm += float(row["precip_mm"]) / 10.0
        for cls, dfm in dead.items():
            dfm.update(ts.year, ts.month, ts.day, ts.hour, ts.minute, 0,
                       float(row["temp_C"]), float(row["hr_pct"]) / 100.0,
                       float(row["rad_Wm2"]), rain_cm)
        day = ts.date()
        if day != last_day:
            sub = hourly[hourly.index.date == day]
            tavg = sub["temp_C"].mean()
            rhavg = sub["hr_pct"].mean()
            tmax = sub["tmax_C"].max();     tmax = sub["temp_C"].max() if pd.isna(tmax) else tmax
            tmin = sub["tmin_C"].min();     tmin = sub["temp_C"].min() if pd.isna(tmin) else tmin
            rhmin = sub["hrmin_pct"].min(); rhmin = sub["hr_pct"].min() if pd.isna(rhmin) else rhmin
            tsd = int(pd.Timestamp(day).timestamp()); jd = day.timetuple().tm_yday
            for lfm in (herb, woody):
                lfm.Update(c_to_f(tavg), c_to_f(tmax), c_to_f(tmin),
                           rhavg, rhmin, jd, float(sub["precip_mm"].sum()), tsd)
            last_day = day

        # ← ÚNIC CANVI: només guardem quan és 23 UTC del dia objectiu
        if day == target_date and ts.hour == TARGET_HOUR:
            res = {c: dead[c].meanWtdMoisture() * 100 for c in dead}
            res["lfmc_herba"]   = herb.CalcRunningAvgHerbFM()
            res["lfmc_llenyos"] = woody.CalcRunningAvgWoodyFM()

    return res or None

# ──────────────────────────────────────────────────────────────────────────────
# INTERPOLACIÓ IDW
# ──────────────────────────────────────────────────────────────────────────────
def idw_interpolation(x, y, z, xi, yi, power=2, k=6):
    xy_st   = np.vstack([x, y]).T
    xy_grid = np.vstack([xi.ravel(), yi.ravel()]).T
    tree = cKDTree(xy_st)
    dist, idx = tree.query(xy_grid, k=k)
    dist[dist == 0] = 1e-12
    w = 1 / (dist ** power)
    w /= w.sum(axis=1, keepdims=True)
    result = np.sum(z[idx] * w, axis=1)
    return result.reshape(xi.shape)

# ──────────────────────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────────────────────
def main():
    target = TARGET_DATE
    meta = pd.read_parquet(meta_path(target)).set_index("station_code")

    codes = stations_available(target)
    print(f"Estacions amb dades el {target}: {len(codes)}")

    start = max(target - timedelta(days=WARMUP_DAYS), DATA_START)
    print(f"Finestra de warm-up: {start} -> {target}  ({(target - start).days + 1} dies)")

    rows = []
    for code in codes:
        if code not in meta.index:
            continue
        lat = float(meta.loc[code, "latitude"]); lon = float(meta.loc[code, "longitude"])
        h = load_station_hourly(code, start, target)
        if h is None:
            continue
        fm = compute_station(h, lat, target)
        if fm is None:
            continue
        rows.append({"station": code, "latitude": lat, "longitude": lon, **fm})

    df = pd.DataFrame(rows)
    print(f"Estacions calculades: {len(df)}")
    print(df[["station","mc1","mc10","mc100","lfmc_herba","lfmc_llenyos"]].describe().round(1))

    # ── Interpolació IDW a graella ICON-EU + retallat Catalunya ───────────
    variables = ["mc1", "mc10", "mc100", "mc1000", "lfmc_herba", "lfmc_llenyos"]
    grids = {}
    for var in variables:
        raw = idw_interpolation(
            df["longitude"].values, df["latitude"].values,
            df[var].values, lon_grid, lat_grid,
        )
        grids[var] = np.where(mask, raw, np.nan)

    # ── Guardar NetCDF ───────────────────────────────────────────────────
    ds = xr.Dataset(
        {var: (["lat", "lon"], grids[var]) for var in variables},
        coords={
            "lat": LATS_ICON,
            "lon": LONS_ICON,
        },
        attrs={
            "title": "ONOFRE fuel moisture — SMC stations IDW",
            "date": str(target),
            "hour_utc": TARGET_HOUR,
            "source": "nfdrs4 (Nelson dead + GSI live)",
            "grid": "ICON-EU 0.0625deg",
            "units": "percent",
        },
    )
    nc_path = f"{ONOFRE_ESTAT}/onofre_fm_{target}_{TARGET_HOUR:02d}utc.nc"
    ds.to_netcdf(nc_path)
    print(f"✓ NetCDF desat: {nc_path}")

    import matplotlib.pyplot as plt
    import geopandas as gpd

    comarques = gpd.read_file("/home/labfire/data/GRAF_MAPS/static_data/comarques/comarques_cat.shp").to_crs(4326)

    panels = [
        ("mc1",          "mc1 (1h, %)",       "RdYlGn",   0,  30),
        ("mc10",         "mc10 (10h, %)",      "RdYlGn",   0,  30),
        ("mc100",        "mc100 (100h, %)",    "RdYlGn",   0,  30),
        ("lfmc_herba",   "LFMC herbaci (%)",   "Spectral", 30, 200),
        ("lfmc_llenyos", "LFMC llenyós (%)",   "Spectral", 30, 200),
    ]

    fig, axes = plt.subplots(1, 5, figsize=(24, 5))

    for ax, (var, title, cm, vmin, vmax) in zip(axes, panels):
        pcm = ax.pcolormesh(
            ds.lon, ds.lat, ds[var].values,
            cmap=cm, shading="auto", vmin=vmin, vmax=vmax,
        )
        comarques.boundary.plot(ax=ax, color="black", linewidth=0.5)
        ax.set_title(title, fontsize=10)
        ax.set_xlim(0.1, 3.4); ax.set_ylim(40.5, 42.95)
        ax.set_aspect(1 / np.cos(np.radians(41.7)))
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(pcm, ax=ax, shrink=0.7, pad=0.02)

    fig.suptitle(f"ONOFRE — Humitat de combustible · {TARGET_DATE} {TARGET_HOUR} UTC · IDW ICON-EU",
                fontsize=13)
    fig.tight_layout()
    plt.show()

    return df, ds

df, ds = main()

In [ ]:
"""
ONOFRE — Pronòstic ROS/FLI  (ICON-EU + NFDRS4 + pyrothermel)
=============================================================
Inputs:
  1. NetCDF d'humitat de combustible a les 23 UTC (SMC stations → IDW ICON-EU)
     → mc100, lfmc_herba, lfmc_llenyos es mantenen constants (escala lenta)
  2. NetCDF de superfície ICON-EU (pronòstic horari D+0 / D+1)
     → T2m, RH, wind, radiació, precipitació acumulada
  3. GeoTIFF del mapa de combustibles PREVINCAT (codis Scott & Burgan)

Pipeline per cada timestep del pronòstic:
  - Nelson forward (mc1, mc10) amb meteo ICON-EU — convergeix en 1-10h
  - mc100 constant des de 23 UTC (memòria massa llarga per 48h)
  - LFMC herba/llenyós constants des de 23 UTC (escala diària)
  - Rothermel ROS + FLI Marc per cada píxel cremable
"""
import os
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from pathlib import Path
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import geopandas as gpd
import matplotlib.pyplot as plt

import nfdrs4
import pyrothermel as pt

# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURACIÓ
# ──────────────────────────────────────────────────────────────────────────────
ICON_DATE = date.today()

FM_FILE      = f"{ONOFRE_ESTAT}/onofre_fm_{TARGET_DATE}_{TARGET_HOUR:02d}utc.nc"
ICON_FILE    = f"/home/labfire/data/ICON-EU/ICONEU_surface_{ICON_DATE:%Y%m%d}_00.nc"
FUEL_FILE    = f"{ONOFRE_REF}/PREVINCAT_ICON/combustible_ICON_EU.tif"
OUTPUT_FILE  = f"{ONOFRE_ESTAT}/ONOFRE_rothermel_{ICON_DATE:%Y%m%d}_00.nc"
OUTPUT_PLOTS = ONOFRE_MAPES

H_MARC = 14700.0  # kJ/kg — heat content (Finney et al. 2021)

regions = gpd.read_file(f"{ONOFRE_REF}/RE_bombers/re_bombers_proj.shp")
regions = regions.to_crs("EPSG:4326")
regions_feat = cfeature.ShapelyFeature(regions['geometry'], ccrs.PlateCarree(), facecolor='none')

comarques = gpd.read_file(f"{ONOFRE_REF}/comarques/comarques_cat.shp")
comarques = comarques.to_crs("EPSG:4326")
comarques_feat = cfeature.ShapelyFeature(comarques['geometry'], ccrs.PlateCarree(), facecolor='none')

# ──────────────────────────────────────────────────────────────────────────────
# FLI MARC CASTELLNOU
# ──────────────────────────────────────────────────────────────────────────────
def fli_marc(ros_ms, w_total_kgm2, ffmc_pct):
    if ffmc_pct >= 21.0:
        return 0.0
    frac_consumed = np.clip((21.0 - ffmc_pct) / (21.0 - 3.0), 0.0, 1.0)
    w_consumed    = w_total_kgm2 * frac_consumed
    heat_factor   = 1.0308 - 0.048 * ffmc_pct
    return max(0.0, H_MARC * w_consumed * heat_factor * ros_ms)

# ──────────────────────────────────────────────────────────────────────────────
# UTILITATS PYROTHERMEL
# ──────────────────────────────────────────────────────────────────────────────
def _fuel_exists(code):
    try:
        pt.FuelModel.from_existing(int(code), units_preset='metric')
        return True
    except Exception:
        return False

def rothermel_pixel(fuel, mc1, mc10, mc100, mc_herb, mc_woody, wind10_ms, slope_deg=0.0):
    """Rothermel ROS (m/s) via pyrothermel, amb humitats individuals."""
    moist = pt.MoistureScenario(
        moisture_one_hour        = mc1,
        moisture_ten_hour        = mc10,
        moisture_hundred_hour    = mc100,
        moisture_live_herbaceous = mc_herb,
        moisture_live_woody      = mc_woody,
        fraction_units           = 'fraction',
    )
    run = pt.PyrothermelRun(
        fuel_model      = fuel,
        moisture_scenario = moist,
        wind_speed       = wind10_ms * 3.6,   # m/s → km/h
        units_preset     = 'metric',
        wind_input_mode  = 'ten_meter',
        slope            = slope_deg,
        canopy_cover     = 0.0,
        canopy_height    = 0.5,
        canopy_ratio     = 0.5,
    )
    res = run.run_surface_fire_in_direction_of_max_spread()
    return max(0.0, res['spread_rate'] / 3.6)   # km/h → m/s

# ──────────────────────────────────────────────────────────────────────────────
# PLOT
# ──────────────────────────────────────────────────────────────────────────────

def plot_ROS_gridspec_ICONEU(
    ds_out,
    comarques_feat,
    regions_feat,
    logo_path,
    ncols=8,
    cell_w=5.0,
    cell_h=3.9,
    cb_h=0.6,
    cb_pad=0.35,
    top_pad=0.8,
    dpi=300,
):
    from matplotlib.colors import BoundaryNorm
    import matplotlib.gridspec as gridspec
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    ros_kmh = ds_out["ros"].values * 3.6
    lats = ds_out.latitude.values
    lons = ds_out.longitude.values

    # Només cada 3h (00, 03, 06, ..., 21)
    mask_3h = np.isin(pd.to_datetime(ds_out.time.values).hour, [0, 3, 6, 9, 12, 15, 18, 21])
    idx_3h = np.where(mask_3h)[0]
    ros_kmh = ros_kmh[idx_3h]
    times_sel = ds_out.time.isel(time=idx_3h)
    num_time_steps = len(idx_3h)

    nrows = int(np.ceil(num_time_steps / ncols))
    fig_w = ncols * cell_w
    fig_h = nrows * cell_h + cb_pad + cb_h + top_pad

    fig = plt.figure(figsize=(fig_w, fig_h), dpi=dpi)
    fig.subplots_adjust(top=1 - top_pad / fig_h)

    gs = gridspec.GridSpec(
        nrows=nrows + 2, ncols=ncols, figure=fig,
        height_ratios=[1]*nrows + [cb_pad/cell_h, cb_h/cell_h],
        hspace=0.0, wspace=0.04,
    )

    proj = ccrs.PlateCarree()

    bounds = [0, 0.1, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0]
    cmap = plt.cm.afmhot_r
    norm = BoundaryNorm(bounds, cmap.N)
    mappable = None

    for t in range(nrows * ncols):
        r, c = divmod(t, ncols)
        ax = fig.add_subplot(gs[r, c], projection=proj)

        if t >= num_time_steps:
            ax.set_axis_off()
            continue

        ax.set_extent([0, 3.5, 40.3, 43], crs=proj)
        ax.set_autoscale_on(False)
        ax.set_aspect("equal", adjustable="box")

        ax.add_feature(cfeature.OCEAN,     facecolor="lightblue", zorder=10)
        ax.add_feature(cfeature.COASTLINE, zorder=2)
        ax.add_feature(cfeature.BORDERS,   zorder=3)
        ax.add_feature(regions_feat,   edgecolor="black", linewidth=0.40, zorder=4)
        ax.add_feature(comarques_feat, edgecolor="black", linewidth=0.07, zorder=5)

        mappable = ax.pcolormesh(
            lons, lats, ros_kmh[t],
            cmap=cmap, norm=norm, shading="auto",
            transform=proj, zorder=1,
        )

        t_str = pd.to_datetime(times_sel.isel(time=t).values).strftime("%Y-%m-%d %H")
        ax.set_title(f"{t_str}Z", fontsize=13)
        ax.set_xticks([]); ax.set_yticks([])

    # ── Colorbar ──────────────────────────────────────────────────────────
    fig.add_subplot(gs[nrows, :]).axis("off")

    sub = gs[nrows + 1, :].subgridspec(1, 3, width_ratios=[3, 6, 3], wspace=0.0)
    ax_cb = fig.add_subplot(sub[0, 1]); ax_cb.axis("off")
    cax = ax_cb.inset_axes([0.10, 0.25, 0.80, 0.50])
    cb = fig.colorbar(mappable, cax=cax, orientation="horizontal",
                      ticks=bounds, spacing="uniform")
    cb.set_label("ROS (km/h)", fontsize=12)
    cb.ax.tick_params(labelsize=11)

    # ── Logo ──────────────────────────────────────────────────────────────
    ax_logo = fig.add_subplot(gs[nrows:nrows + 2, -2:])
    ax_logo.axis("off")
    logo = plt.imread(logo_path)
    ax_logo.imshow(logo, aspect="equal")
    ax_logo.set_anchor("E")

    top_frac = top_pad / fig_h
    fig.suptitle(
        "ONOFRE — ROS pronòstic (km/h) — ICON-EU forecast",
        fontsize=22, y=1 - top_frac / 2,
    )
    return fig

# ──────────────────────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────────────────────
def main():
    # ── 1. Carregar dades ────────────────────────────────────────────────
    print("Carregant dades...")
    ds_fm   = xr.open_dataset(FM_FILE)
    ds_icon = xr.open_dataset(ICON_FILE)

    # Retallar a Catalunya
    ds_icon = ds_icon.sel(
        latitude=slice(40.3125, 43.0),
        longitude=slice(0.0, 3.5),
    )

    if "rad_global" not in ds_icon:
        ds_icon["rad_global"] = ds_icon["aswdir_s"] + ds_icon["aswdifd_s"]
        ds_icon["rad_global"].attrs = {
            "long_name": "Downward shortwave radiation (direct + diffuse)",
            "units": "W m-2",
            "note": "Acumulada des del run. Cal diferenciar abans de Nelson.",
        }

    times = ds_icon.time.values
    lats  = ds_icon.latitude.values
    lons  = ds_icon.longitude.values
    nt, ny, nx = len(times), len(lats), len(lons)

    # ── 2. Alinear graella FM → ICON-EU (nearest) ───────────────────────
    ds_fm_a = ds_fm.interp(lat=lats, lon=lons, method="nearest")

    # mc100 i LFMC constants durant tot el pronòstic (% → fracció)
    mc100_const  = np.nan_to_num(ds_fm_a["mc100"].values / 100.0, nan=0.15)
    lfmc_h_const = np.nan_to_num(ds_fm_a["lfmc_herba"].values / 100.0, nan=1.0)
    lfmc_w_const = np.nan_to_num(ds_fm_a["lfmc_llenyos"].values / 100.0, nan=0.8)

    # ── 3. Variables ICON-EU ─────────────────────────────────────────────
    t2m = ds_icon["t_2m"].values.copy()
    if np.nanmean(t2m) > 100:        # Kelvin → °C
        t2m -= 273.15
    rh       = ds_icon["relhum_2m"].values          # %
    wind10   = np.sqrt(ds_icon["u_10m"].values**2 +
                       ds_icon["v_10m"].values**2)   # m/s
    rad      = ds_icon["rad_global"].values          # W/m²
    #rain_cm  = ds_icon["tot_prec"].values / 10.0     # mm acumulat → cm acumulat
    rain_cm = ds_icon["tot_prec"].values.astype(np.float64) / 10.0
    rain_cm = np.maximum.accumulate(rain_cm, axis=0)

    # ── 4. Mapa de combustibles (encaixar en graella ICON-EU) ────────────
    print("Llegint mapa de combustibles...")
    with rasterio.open(FUEL_FILE) as src:
        fuel_raw = src.read(2).astype(np.int16)
        bounds = src.bounds   # left=0.0625, bottom=40.5, right=3.375, top=42.9375

    # El GeoTIFF és north-up (fila 0 = nord) → flipar per lat creixent
    fuel_raw = fuel_raw[::-1]

    # Trobar on encaixa dins la graella ICON-EU
    j0 = np.argmin(np.abs(lons - bounds.left))     # ≈ 1
    i0 = np.argmin(np.abs(lats - bounds.bottom))    # ≈ 3
    fny, fnx = fuel_raw.shape                        # 39, 53

    fuel_map = np.zeros((ny, nx), dtype=np.int16)    # 44×57, tot 0 (no cremable)
    fuel_map[i0:i0+fny, j0:j0+fnx] = fuel_raw

    print(f"Fuel map {fny}×{fnx} encaixat a ICON-EU {ny}×{nx} "
        f"(offset lat={i0}, lon={j0})")

    codes_burnable = [c for c in np.unique(fuel_map) if _fuel_exists(int(c))]
    print(f"Codis combustibles: {codes_burnable}")

    fuel_objects = {c: pt.FuelModel.from_existing(int(c), units_preset='metric')
                    for c in codes_burnable}
    w_total = {
        int(c): (f.fuel_load_one_hour + f.fuel_load_ten_hour +
                f.fuel_load_hundred_hour + f.fuel_load_live_herbaceous +
                f.fuel_load_live_woody)
        for c, f in fuel_objects.items()
    }

    burnable = np.isin(fuel_map, codes_burnable)
    cells = list(zip(*np.where(burnable)))
    print(f"Cel·les cremables: {len(cells)}")

    # ── 5. Inicialitzar Nelson (mc1, mc10) per cada cel·la cremable ──────
    nelson = {}
    for i, j in cells:
        dfm1 = nfdrs4.DeadFuelMoisture()
        dfm1.initDeadFuelMoisture1()

        dfm10 = nfdrs4.DeadFuelMoisture()
        dfm10.initDeadFuelMoisture10()

        mc1_init  = float(ds_fm_a["mc1"].values[i, j]) / 100.0
        mc10_init = float(ds_fm_a["mc10"].values[i, j]) / 100.0

        dfm1.setMoisture(mc1_init)
        dfm10.setMoisture(mc10_init)
        nelson[(i, j)] = (dfm1, dfm10)

    # ── 6. Bucle temporal: Nelson forward + Rothermel ────────────────────
    mc1_out  = np.full((nt, ny, nx), np.nan, dtype=np.float32)
    mc10_out = np.full((nt, ny, nx), np.nan, dtype=np.float32)
    ros_out  = np.zeros((nt, ny, nx), dtype=np.float32)
    fli_out  = np.zeros((nt, ny, nx), dtype=np.float32)

    print(f"Executant Nelson + Rothermel per {nt} timesteps × {len(cells)} cel·les...")

    slope_pct = 10.0
    slope_deg_equivalent = np.degrees(np.arctan(slope_pct / 100.0))  

    for t in range(nt):
        ts = pd.Timestamp(times[t])

        for i, j in cells:
            # ── Nelson update ──
            dfm1, dfm10 = nelson[(i, j)]
            T   = float(t2m[t, i, j])
            #RH  = float(rh[t, i, j]) / 100.0    # % → fracció
            RH = np.clip(float(rh[t, i, j]) / 100.0, 0.001, 0.999)
            R   = float(rad[t, i, j])
            P   = float(rain_cm[t, i, j])        # acumulat (cm)

            dfm1.update(ts.year, ts.month, ts.day, ts.hour, ts.minute, 0,
                        T, RH, R, P)
            dfm10.update(ts.year, ts.month, ts.day, ts.hour, ts.minute, 0,
                         T, RH, R, P)

            mc1_frac  = dfm1.meanWtdMoisture()
            mc10_frac = dfm10.meanWtdMoisture()

            mc1_out[t, i, j]  = mc1_frac * 100.0   # guardem en %
            mc10_out[t, i, j] = mc10_frac * 100.0

            # ── Rothermel ──
            code = int(fuel_map[i, j])
            fuel = fuel_objects[code]
            wt   = w_total[code]
            w    = float(wind10[t, i, j])

            mc100_frac = float(mc100_const[i, j])
            lh_frac    = float(lfmc_h_const[i, j])
            lw_frac    = float(lfmc_w_const[i, j])

            ros_ms = rothermel_pixel(
                fuel, mc1_frac, mc10_frac, mc100_frac,
                lh_frac, lw_frac, w,  slope_deg=slope_deg_equivalent
            )
            ros_out[t, i, j] = ros_ms
            fli_out[t, i, j] = fli_marc(ros_ms, wt, mc1_frac * 100.0)

        # Diagnòstic per tipus de combustible (afegir dins for t, després de tots els píxels)
        if t == 18:
            for cls_name, cls_code in [("GR2",102), ("GS3",123), ("SH6",146), ("TU3",163), ("TL2",182)]:
                mask_cls = fuel_map == cls_code
                r_cls = ros_out[t][mask_cls]
                if r_cls.max() > 0:
                    idx_max = np.unravel_index(
                        np.argmax(np.where(mask_cls, ros_out[t], 0)), ros_out[t].shape)
                    print(f"    {cls_name}: ROS mean={r_cls[r_cls>0].mean():.3f} "
                        f"max={r_cls.max():.3f} m/s  (n={np.sum(r_cls>0)})  "
                        f"wind={wind10[t, idx_max[0], idx_max[1]]:.1f} m/s  "
                        f"mc1={mc1_out[t, idx_max[0], idx_max[1]]:.1f}%")

        # Diagnosi per hora
        r_pos = ros_out[t][ros_out[t] > 0]
        f_pos = fli_out[t][fli_out[t] > 0]
        print(f"  {t+1:02d}/{nt}  {ts}  "
              f"ROS={r_pos.mean():.3f} m/s  FLI={f_pos.mean():.0f} kW/m"
              if len(r_pos) > 0 else
              f"  {t+1:02d}/{nt}  {ts}  ROS=0  FLI=0")

    # ── 7. Guardar NetCDF ────────────────────────────────────────────────
    coords = {"time": times, "latitude": lats, "longitude": lons}
    ds_out = xr.Dataset({
        "ros": (["time", "latitude", "longitude"], ros_out, {
            "long_name": "Rate of Spread (Rothermel 1972)",
            "units": "m s-1"}),
        "fli": (["time", "latitude", "longitude"], fli_out, {
            "long_name": "Fire Line Intensity (Castellnou)",
            "units": "kW m-1"}),
        "mc1": (["time", "latitude", "longitude"], mc1_out, {
            "long_name": "1-hour dead fuel moisture (Nelson)",
            "units": "percent"}),
        "mc10": (["time", "latitude", "longitude"], mc10_out, {
            "long_name": "10-hour dead fuel moisture (Nelson)",
            "units": "percent"}),
    }, coords=coords, attrs={
        "title": "ONOFRE — Pronòstic ROS/FLI",
        "fuel_moisture_init": FM_FILE,
        "icon_eu_forecast": ICON_FILE,
        "mc100_source": "23 UTC constant (Nelson no convergeix en 48h)",
        "lfmc_source": "23 UTC constant (escala diària)",
        "mc1_mc10_source": "Nelson forward amb meteo ICON-EU",
    })
    ds_out.to_netcdf(OUTPUT_FILE)
    print(f"\n✓ Guardat: {OUTPUT_FILE}")

    import matplotlib.pyplot as plt
    SHP = "/home/labfire/data/GRAF_MAPS/static_data/comarques/comarques_cat.shp"
    #animate_ros(ds_out, OUTPUT_FILE.replace(".nc", ".mp4"), comarques_shp=SHP)

    fig = plot_ROS_gridspec_ICONEU(
        ds_out         = ds_out,
        comarques_feat = comarques_feat,
        regions_feat   = regions_feat,
        logo_path      = f"{ONOFRE_REF}/logo_bombers.png",
        )
    plt.show()
    date_str = pd.Timestamp(ds_out.time.values[0]).strftime("%Y%m%d")
    #plt.savefig(os.path.join(OUTPUT_PLOTS, f"ROS_{date_str}.png"),
                #dpi=300, bbox_inches="tight", pad_inches=0.12)

    return ds_out

ds = main()

In [ ]:
# ============================================================
#   EXPORTA FRAMES PEL VISOR WEB (ROS/FLI)
#   → només cobertura Catalunya (humitat de combustible SMC).
# ============================================================
import sys
sys.path.insert(0, os.path.join(ONOFRE_REPO, "generator"))  # ajusta si cal
from generate_frames import generate_variable, build_land_mask, apply_polygon_mask

run_tag_ros = pd.Timestamp(ds.time.values[0]).strftime("%Y%m%d_%H")

nt_ros = ds.dims["time"]
ros_list  = [ds["ros"].isel(time=t).values * 3.6 for t in range(nt_ros)]  # m/s → km/h
times_ros = [pd.Timestamp(t) for t in ds.time.values]

inside_land_ros = build_land_mask(ds.longitude.values, ds.latitude.values, resolution="10m")
ros_list_masked = apply_polygon_mask(ros_list, inside_land_ros)

# Reutilitza la teva funció de mosaic tal qual (signatura diferent, per això
# es genera aquí la figura i es passa ja feta a generate_variable)
fig_ros = plot_ROS_gridspec_ICONEU(
    ds_out         = ds,
    comarques_feat = comarques_feat,
    regions_feat   = regions_feat,
    logo_path      = f"{ONOFRE_REF}/logo_bombers.png",
)

generate_variable(
    "ros_fli", ros_list_masked, ds.longitude.values, ds.latitude.values, times_ros, run_tag_ros,
    mosaic_fig=fig_ros,
)
plt.close(fig_ros)

print(f"✓ Frames exportats pel visor web (run {run_tag_ros}): ROS/FLI")
